In [3]:
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
from scipy.stats import spearmanr
import networkx as nx
import warnings
import re
import os
warnings.filterwarnings("ignore")


output_dir = "Graphical_Lasso_Results"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


# ============================================================
# 0. Read data and define columns
# ============================================================
CSV_PATH = r"gist_data2_4alldata1.xlsx"  # Adjust path as needed
df = pd.read_excel(CSV_PATH)

def clean_col(c):
    c = str(c).strip().replace("\u200b", "")
    return re.sub(r"\s+", "", c)

df.columns = [clean_col(c) for c in df.columns]

POSTOP_COLS = ["CD34", "CD117", "PDGFR-α", "Dog-1", "S-100", "SMA", "KI-67"]
Y_MUL_NAME = "RiskCategory_encoded"

preop_cols = [c for c in df.columns if c not in POSTOP_COLS + [Y_MUL_NAME]]
postop_cols = [c for c in POSTOP_COLS if c in df.columns]

# 构造数值标签
df["_Y_mul_str"] = df[Y_MUL_NAME].astype(str).str.strip()
df["_Y_mul_num"] = pd.Categorical(df["_Y_mul_str"]).codes.astype(int)

# ============================================================
# 1. Prepare features and target (use all data)
# ============================================================
X_full = df[preop_cols + postop_cols].copy()
y_full = df[Y_MUL_NAME].copy()
if y_full.dtype == 'object':
    y_full = pd.Categorical(y_full).codes

print(f"Total samples: {X_full.shape[0]}")

# ============================================================
# 2. Preprocessing: one‑hot encoding + standardisation (on full data)
# ============================================================
num_cols = [c for c in X_full.columns if pd.api.types.is_numeric_dtype(X_full[c])]
cat_cols = [c for c in X_full.columns if c not in num_cols]

X_enc = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)
# Remove constant features (zero variance)
non_const = X_enc.columns[X_enc.std() > 0]
X_enc = X_enc[non_const]

scaler = StandardScaler()
X_z = scaler.fit_transform(X_enc)
feature_names = X_enc.columns.tolist()
print(f"Number of features after preprocessing: {len(feature_names)}")

# ============================================================
# 3. Graphical Lasso estimation (on full data)
# ============================================================
gl = GraphicalLassoCV(
    alphas=4,
    cv=5,
    n_refinements=4,
    tol=1e-4,
    max_iter=100,
    mode='cd',
)
gl.fit(X_z)

print(f"Optimal regularisation parameter λ (alpha): {gl.alpha_:.6f}")
print("λ selection method: 5‑fold cross‑validation (GraphicalLassoCV)")
print("Algorithm: coordinate descent (mode='cd')")
print(f"Convergence tolerance: tol=1e-4, max_iter=100")

Theta = gl.precision_
d = np.sqrt(np.diag(Theta))
partial_corr = -Theta / (np.outer(d, d) + 1e-12)
np.fill_diagonal(partial_corr, 0)

# ============================================================
# 4. Node strength (replaces "precision‑matrix importance")
# ============================================================
node_strength = {}
for i, feat in enumerate(feature_names):
    strength = np.sum(np.abs(partial_corr[i, :]))
    node_strength[feat] = strength
df_strength = pd.DataFrame(list(node_strength.items()), columns=["feature", "node_strength"])

# ============================================================
# 5. Spearman correlation with risk category
# ============================================================
spearman_dict = {}
for i, feat in enumerate(feature_names):
    x_col = X_z[:, i]
    if np.std(x_col) < 1e-8:
        continue
    rho, _ = spearmanr(x_col, y_full)
    spearman_dict[feat] = abs(rho)
df_spearman = pd.DataFrame(list(spearman_dict.items()), columns=["feature", "Correlation coefficient"])

# ============================================================
# 6. PageRank based on the partial correlation network
# ============================================================
G = nx.Graph()
G.add_nodes_from(feature_names)
for i in range(len(feature_names)):
    for j in range(i+1, len(feature_names)):
        w = abs(partial_corr[i, j])
        if w > 1e-6:
            G.add_edge(feature_names[i], feature_names[j], weight=w)

pagerank_dict = nx.pagerank(G, weight='weight')
df_pagerank = pd.DataFrame(list(pagerank_dict.items()), columns=["feature", "pagerank"])

# ============================================================
# 7. Merge the three metrics and normalise
# ============================================================
df_merged = df_spearman.merge(df_strength, on='feature', how='inner')
df_merged = df_merged.merge(df_pagerank, on='feature', how='inner')
print(f"Valid features after merging: {len(df_merged)}")

for metric in ['Correlation coefficient', 'node_strength', 'pagerank']:
    minv, maxv = df_merged[metric].min(), df_merged[metric].max()
    df_merged[f'{metric}_norm'] = (df_merged[metric] - minv) / (maxv - minv)

# ============================================================
# 8. Random weight sensitivity analysis (Dirichlet, 500 draws)
# ============================================================
np.random.seed(42)
n_random = 500
random_weights = np.random.dirichlet(alpha=(1,1,1), size=n_random)

df_merged = df_merged.reset_index(drop=True)

stability_counts = {feat: 0 for feat in df_merged['feature']}
for w in random_weights:
    w1, w2, w3 = w
    score = (w1 * df_merged['Correlation coefficient_norm'] +
             w2 * df_merged['node_strength_norm'] +
             w3 * df_merged['pagerank_norm'])
    sorted_idx = score.sort_values(ascending=False).index
    top10 = df_merged.loc[sorted_idx, 'feature'].head(10).tolist()
    for feat in top10:
        stability_counts[feat] += 1

df_stability = pd.DataFrame(list(stability_counts.items()), columns=['feature', 'count'])
df_stability['prob'] = df_stability['count'] / n_random
df_stability = df_stability.sort_values('prob', ascending=False).reset_index(drop=True)

print("\nFeature stability (probability of being in top‑10 across 500 random weight draws):")
print(df_stability.head(10).to_string(index=False))

# ============================================================
# 9. Select final top‑10 features and merge with metrics
# ============================================================
selected_feature = df_stability.head(10)['feature'].tolist()

# Merge original metrics
df_stability = df_stability.merge(
    df_merged[['feature', 'Correlation coefficient', 'pagerank', 'node_strength']],
    on='feature', how='left'
)
df_stability = df_stability.sort_values('prob', ascending=False).reset_index(drop=True)

# Save results
df_stability.to_csv(os.path.join(output_dir, "feature_stability_with_metrics.csv"), index=False, encoding='utf-8-sig')


Total samples: 433
Number of features after preprocessing: 18
Optimal regularisation parameter λ (alpha): 0.070863
λ selection method: 5‑fold cross‑validation (GraphicalLassoCV)
Algorithm: coordinate descent (mode='cd')
Convergence tolerance: tol=1e-4, max_iter=100
Valid features after merging: 18

Feature stability (probability of being in top‑10 across 500 random weight draws):
        feature  count  prob
          KI_67    500   1.0
          Dog-1    500   1.0
ShapeRegularity    500   1.0
     Ulceration    500   1.0
          CD117    500   1.0
